# E3 — LSTM, GRU và các đối chứng trên chuỗi ảnh

LSTM và GRU nhận cùng ảnh Fashion-MNIST dưới dạng 28 hàng hoặc 49 patch 4×4 theo thứ tự quét từ trái sang phải, trên xuống dưới. MLP và CNN của E1 là hai đối chứng không hồi quy. Bốn cấu hình hồi tiếp đối chiếu loại cổng và cách chuyển ảnh thành chuỗi.

Giao thức chính gồm 30 epoch cho mỗi cấu hình và ba seed 42, 43, 44; mỗi lượt chọn checkpoint trên tập xác thực. Số liệu trong notebook được tính từ lượt chạy này. Cần cài gói dự án với `python -m pip install --no-deps -e .`, các phụ thuộc trong `requirements.txt` và kernel Python có `ipykernel`. Xem [giao thức thực nghiệm](../docs/EXPERIMENTS.md) và [hướng dẫn sử dụng AI](../AI_USE_REPORT.md).

## Thiết lập lượt chạy

Có thể sửa giá trị mặc định trong ô dưới hoặc đặt biến môi trường tương ứng với tiền tố `CO5085_EXERCISES_`. Để chạy theo seed, đặt `SEEDS=42`; để kiểm tra nhanh, đặt `MODELS` thành một mã cấu hình, `EPOCHS=1` và chọn `OUTPUT_ROOT` riêng. Các phép so sánh thời gian cần cùng thiết bị, số luồng và số worker.

In [ ]:
from pathlib import Path
import subprocess
import sys

colab_root = Path('/content/CO5085-deep-learning')
roots = (Path.cwd(), *Path.cwd().parents, colab_root)
ROOT = next((path for path in roots if (path / 'configs/experiments.json').is_file()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/dathuynh1108/CO5085-deep-learning.git', str(colab_root)], check=True)
    ROOT = colab_root
if ROOT is None:
    raise FileNotFoundError('Mở notebook từ thư mục CO5085-deep-learning.')
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display

from common.config import get_experiment, load_experiments
from common.data import CLASS_NAMES, prepare_data
from common.engine import train
from common.evaluation import evaluate_run
from common.factory import build_model, parameter_count
from common.runtime import atomic_csv, atomic_json, choose_device, environment_info, seed_everything
from common.tokenization import image_sequence
from exercise_notebook import (
    flag, read_json, setting, show_comparison, show_figure, show_results, verified_training_status,
)

EXERCISE = 'E3'

In [ ]:
SEEDS = tuple(int(value) for value in setting('SEEDS', '42,43,44').split(','))
MODELS = setting('MODELS')                       # Ví dụ: e1_mlp,e1_cnn
EPOCHS = int(setting('EPOCHS')) if setting('EPOCHS') else None
DEVICE = setting('DEVICE', 'auto')               # auto, cpu, cuda, cuda:0
WORKERS = int(setting('WORKERS', '0'))           # Đặt 8 để khớp môi trường GPU lịch sử
DATA_ROOT = (ROOT / setting('DATA_ROOT', 'data')).resolve()
OUTPUT_ROOT = (ROOT / setting('OUTPUT_ROOT', 'runs/notebooks/exercises')).resolve()
RESUME = flag('RESUME')
DETERMINISTIC = flag('DETERMINISTIC')
DOWNLOAD = flag('DOWNLOAD', default=True)
CONFIRM_TEST = flag('CONFIRM_TEST')               # Chỉ bật sau khi chốt giao thức

if not SEEDS or len(set(SEEDS)) != len(SEEDS) or any(not 0 <= seed < 2**32 for seed in SEEDS):
    raise ValueError('SEEDS phải gồm các seed phân biệt trong [0, 2**32).')
SEEDS = tuple(sorted(SEEDS))
if EPOCHS is not None and OUTPUT_ROOT == (ROOT / 'runs/notebooks/exercises').resolve():
    raise ValueError('Thay EPOCHS cần OUTPUT_ROOT riêng để tách lượt kiểm tra.')
if OUTPUT_ROOT.is_relative_to((ROOT / 'runs' / 'exercises').resolve()):
    raise ValueError('Chọn OUTPUT_ROOT khác thư mục kết quả lịch sử runs/exercises.')
if WORKERS < 0:
    raise ValueError('WORKERS phải là số nguyên không âm.')
if CONFIRM_TEST and EPOCHS is not None:
    raise ValueError('Lượt thay đổi EPOCHS chỉ đánh giá trên tập xác thực.')
device = choose_device(DEVICE)

presets = load_experiments()
allowed = {row.id for row in presets if row.exercise == EXERCISE and row.group == 'required'}
allowed.update({'e1_mlp', 'e1_cnn'})
selected = tuple(value.strip() for value in MODELS.split(',')) if MODELS else tuple(
    row.id for row in presets if row.id in allowed
)
if len(set(selected)) != len(selected) or set(selected) - allowed:
    raise ValueError(f'MODELS phải gồm các mã phân biệt thuộc: {sorted(allowed)}')
configs = [get_experiment(identifier, epochs=EPOCHS) for identifier in selected]
display({'thiết bị': str(device), 'seed': SEEDS, 'worker': WORKERS, 'kết quả': str(OUTPUT_ROOT)})
display([{key: config.to_dict()[key] for key in (
    'id', 'epochs', 'batch_size', 'lr', 'tokenizer', 'backend', 'group',
)} for config in configs])

## Khảo sát dữ liệu Fashion-MNIST

Chia phân tầng 90/10 từ tập phát hành dành cho huấn luyện với seed 2026. Giá trị trung bình và độ lệch chuẩn pixel được tính trên các chỉ số train. Biểu đồ nhãn và ảnh minh họa chỉ dùng train/xác thực; tập kiểm tra giữ lại cho đánh giá sau khi chốt giao thức.

In [ ]:
split_seeds = {config.split_seed for config in configs}
if len(split_seeds) != 1:
    raise ValueError('So sánh yêu cầu cùng phép chia dữ liệu.')
train_set, validation_set, dataset = prepare_data(DATA_ROOT, split_seeds.pop(), DOWNLOAD)
train_counts = np.bincount(train_set.labels.numpy()[train_set.indices], minlength=len(CLASS_NAMES))
validation_counts = np.bincount(validation_set.labels.numpy()[validation_set.indices], minlength=len(CLASS_NAMES))
display({key: dataset[key] for key in ('n_train', 'n_validation', 'mean', 'std', 'fingerprint')})

fig, ax = plt.subplots(figsize=(10, 3.5))
labels = np.arange(len(CLASS_NAMES))
ax.bar(labels - .18, train_counts, .36, label='Huấn luyện')
ax.bar(labels + .18, validation_counts, .36, label='Xác thực')
ax.set_xticks(labels, CLASS_NAMES, rotation=30, ha='right')
ax.set(ylabel='Số ảnh', title='Phân bố nhãn theo tập')
ax.legend()
fig.tight_layout()
show_figure(fig)

In [ ]:
rng = np.random.default_rng(2026)
y = train_set.labels.numpy()
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for cls, ax in enumerate(axes.flat):
    index = int(rng.choice(train_set.indices[y[train_set.indices] == cls]))
    ax.imshow(train_set.images[index], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{CLASS_NAMES[cls]} · {index}', fontsize=9)
    ax.axis('off')
fig.tight_layout()
show_figure(fig)

## Cấu trúc mô hình

Hai biểu diễn có dạng $(T,D)=(28,28)$ và $(49,16)$. LSTM và GRU có một tầng hồi tiếp một chiều với 128 đơn vị ẩn. Bộ phân loại nhận trạng thái ẩn ở bước cuối của chuỗi.

In [ ]:
seed_everything(SEEDS[0], DETERMINISTIC)
images = torch.stack([train_set[i][0] for i in range(4)])
model_shapes = []
with torch.inference_mode():
    for config in configs:
        model = build_model(config).eval()
        row = {
            'experiment': config.id,
            'parameters': parameter_count(model),
            'logits': list(model(images).shape),
        }
        if config.exercise == 'E3':
            sequence = image_sequence(images, config.tokenizer, config.patch_size)
            row['sequence_shape'] = ' × '.join(map(str, sequence.shape[1:]))
        model_shapes.append(row)
display(model_shapes)

## Huấn luyện và chọn checkpoint

Mỗi batch trong [vòng lặp PyTorch dùng chung](../src/common/engine.py) đặt gradient về 0, tính logits và cross-entropy, lan truyền ngược, cắt chuẩn gradient và cập nhật AdamW. Scheduler cosine cập nhật sau mỗi epoch. Độ chính xác xác thực chọn checkpoint; nếu bằng nhau, chọn cross-entropy xác thực nhỏ hơn. Mỗi thư mục run lưu cấu hình, seed, fingerprint dữ liệu/mã nguồn, lịch sử epoch, trọng số tốt nhất và trạng thái dùng để tiếp tục huấn luyện.


Nếu kết quả E1 cùng seed đã có trong `OUTPUT_ROOT`, notebook kiểm tra chữ ký trước khi dùng MLP/CNN làm đối chứng. Nếu chưa có, hai đối chứng được huấn luyện tại đây theo cùng giao thức.

In [ ]:
run_environment = environment_info(device)
directories = {}
for config in configs:
    for seed in SEEDS:
        directory = OUTPUT_ROOT / config.id / f'seed_{seed}'
        status = verified_training_status(
            directory, config, seed, dataset, WORKERS, DETERMINISTIC, run_environment,
        )
        if status == 'trained':
            print(f'Tái sử dụng lượt đã hoàn tất: {config.id}, seed={seed}', flush=True)
        else:
            if status == 'incomplete' and not RESUME:
                raise RuntimeError(f'Lượt chạy chưa hoàn tất: {directory}. Bật RESUME để tiếp tục.')
            directory = train(
                config, seed, DATA_ROOT, OUTPUT_ROOT, device_name=DEVICE, workers=WORKERS,
                resume=status == 'incomplete', deterministic=DETERMINISTIC, download=DOWNLOAD,
            )
        directories[(config.id, seed)] = directory

## Đánh giá và phân tích lỗi

Mặc định, checkpoint đã chọn được đánh giá trên tập xác thực. Sau khi cố định mô hình và thiết lập, bật `CONFIRM_TEST` để đánh giá trên tập kiểm tra giữ lại. Bảng dùng tất cả các seed đã chọn; độ lệch chuẩn mẫu chỉ có khi có ít nhất hai seed. Đường cong và ma trận dùng seed nhỏ nhất; mỗi thư mục run lưu ma trận số đếm và tối đa tám lỗi có độ tin cậy cao.

In [ ]:
split = 'test' if CONFIRM_TEST else 'validation'
report_dir = OUTPUT_ROOT / 'notebook_analysis' / EXERCISE / ('seeds_' + '_'.join(map(str, SEEDS))) / split
evaluations, results = {}, []
for (identifier, seed), directory in directories.items():
    result = evaluate_run(directory, DATA_ROOT, split=split, confirm_test=CONFIRM_TEST,
                          device_name=DEVICE, workers=WORKERS, download=DOWNLOAD)
    evaluations[(identifier, seed)] = result
    training = read_json(directory / 'metrics.json')
    results.append({
        'experiment': identifier, 'seed': seed, 'split': split, 'best_epoch': result['best_epoch'],
        'accuracy': result['accuracy'], 'macro_f1': result['macro_f1'],
        'cross_entropy': result['cross_entropy'], 'n_parameters': training['n_parameters'],
        'median_train_seconds': training['median_train_seconds'],
        'total_train_seconds': training['total_train_seconds'],
        'inference_images_per_second': result['inference_images_per_second'],
        'signature': result['signature'], 'model_sha256': result['model_sha256'],
    })
atomic_csv(report_dir / 'results.csv', results)
atomic_json(report_dir / 'protocol.json', {
    'notebook': f'{EXERCISE}.ipynb', 'configurations': [config.to_dict() for config in configs],
    'seeds': list(SEEDS), 'dataset': dataset, 'device': str(device), 'workers': WORKERS,
    'deterministic': DETERMINISTIC, 'split': split, 'confirm_test': CONFIRM_TEST,
})
show_results(results)

In [ ]:
representative_seed = min(SEEDS)
show_comparison(configs, directories, evaluations, representative_seed, report_dir)
reference = configs[0].id
examples = directories[(reference, representative_seed)] / 'plots' / f'mistakes_{split}.png'
if examples.is_file():
    print(f'Lỗi có độ tin cậy cao: {reference}, seed={representative_seed}')
    display(Image(filename=str(examples), width=650))

## Chẩn đoán phụ thuộc vào thứ tự chuỗi

Trên tối đa 256 ảnh xác thực có chỉ số gốc nhỏ nhất, so sánh thứ tự gốc, đảo ngược và một hoán vị cố định cho bốn cấu hình LSTM và GRU. Gradient được tính qua các trạng thái trung gian của phép khai triển hồi quy. Đảo và hoán vị làm thay đổi phân bố đầu vào; độ giảm độ chính xác không tự chứng minh khả năng ghi nhớ dài hạn.

In [ ]:
from E3.diagnose import diagnose

targets = {config.id for config in configs if config.exercise == EXERCISE}
diagnostic_rows = []
for (identifier, seed), directory in directories.items():
    if identifier not in targets:
        continue
    diagnose(directory, DATA_ROOT, DEVICE, max_samples=256, batch_size=32)
    diagnostic = read_json(directory / 'diagnostics' / 'summary.json')
    diagnostic_rows.append({'experiment': identifier, 'seed': seed,
                            'n_samples': diagnostic['n_samples'], **diagnostic['accuracy']})
    if seed == representative_seed:
        print(f'Gradient theo bước chuỗi: {identifier}, seed={seed}')
        display(Image(filename=str(directory / 'diagnostics' / 'temporal_gradients.png'), width=650))
if diagnostic_rows:
    atomic_csv(report_dir / 'sequence_order.csv', diagnostic_rows)
    display(diagnostic_rows)

## Cách đọc kết quả

Đối chiếu LSTM/GRU theo cùng biểu diễn, rồi đối chiếu hàng/patch theo cùng loại cell. So sánh thêm với MLP và CNN để đánh giá ảnh hưởng của cách biểu diễn ảnh. Chuỗi 28/49 bước giới hạn phạm vi kết luận về phụ thuộc dài hạn.